In [2]:
from datasets import load_dataset
from tqdm import cli

dataset = load_dataset("maartengr/arxiv_nlp")

dataset

DatasetDict({
    train: Dataset({
        features: ['Titles', 'Abstracts', 'Years', 'Categories'],
        num_rows: 44949
    })
})

In [3]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from umap import UMAP  # 依赖 uv add umap-learn
from hdbscan import HDBSCAN # 依赖 uv add hdbscan

dataset = load_dataset("maartengr/arxiv_nlp")["train"] # [Titles, Abstracts, Years, Categories]

abstracts = dataset["Abstracts"]
titles = dataset["Titles"]

# text embeddings
embedding_model = SentenceTransformer("thenlper/gte-small", device="mps")
embeddings = embedding_model.encode(abstracts, batch_size=10, device="mps", show_progress_bar=True)
print(embeddings.shape)  # (44949, 384)

# dimensionality reduction, 从 384 降到 5
umap_model = UMAP(n_components=5, min_dist=0.0, metric="cosine", random_state=42)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/4495 [00:00<?, ?it/s]

(44949, 384)


In [4]:
from hdbscan import HDBSCAN
# We fit the model and extract the clusters
hdbscan_model = HDBSCAN(
    min_cluster_size=50, metric="euclidean",
    cluster_selection_method="eom"
)

In [5]:
from bertopic import BERTopic, representation

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True,
).fit(abstracts, embeddings)

2026-05-15 15:54:37,478 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-15 15:55:00,163 - BERTopic - Dimensionality - Completed ✓
2026-05-15 15:55:00,164 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-15 15:55:01,869 - BERTopic - Cluster - Completed ✓
2026-05-15 15:55:01,874 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-15 15:55:03,176 - BERTopic - Representation - Completed ✓


In [6]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,14252,-1_the_and_of_to,"[the, and, of, to, in, we, for, that, language...",[ The task of written language identification...
1,0,2207,0_speech_asr_recognition_end,"[speech, asr, recognition, end, acoustic, audi...",[ Speech-enabled systems typically first conv...
2,1,1268,1_medical_clinical_biomedical_patient,"[medical, clinical, biomedical, patient, notes...",[ Biomedical Named Entity Recognition (NER) i...
3,2,991,2_translation_nmt_machine_bleu,"[translation, nmt, machine, bleu, neural, engl...",[ The quality of machine translation is rapid...
4,3,891,3_summarization_summaries_summary_abstractive,"[summarization, summaries, summary, abstractiv...","[ In this paper, we present a model for gener..."
...,...,...,...,...,...
149,148,53,148_reviews_opinion_summaries_summarization,"[reviews, opinion, summaries, summarization, r...",[ When faced with a large number of product r...
150,149,52,149_translation_indian_hindi_machine,"[translation, indian, hindi, machine, smt, eng...",[ We present in this paper our work on compar...
151,150,51,150_moe_experts_mixture_routing,"[moe, experts, mixture, routing, expert, spars...",[ The Mixture of Experts (MoE) models are an ...
152,151,51,151_multimodal_sentiment_modality_fusion,"[multimodal, sentiment, modality, fusion, moda...",[ Multimodal machine learning is a core resea...


In [7]:
topic_model.get_topic(0)

[('speech', np.float64(0.028708255840916314)),
 ('asr', np.float64(0.019044412757881737)),
 ('recognition', np.float64(0.013632010750850288)),
 ('end', np.float64(0.010173507591108298)),
 ('acoustic', np.float64(0.00968876818453583)),
 ('audio', np.float64(0.006949467627675897)),
 ('speaker', np.float64(0.006930496386245398)),
 ('the', np.float64(0.006427940485342758)),
 ('wer', np.float64(0.006420502697320617)),
 ('error', np.float64(0.006408474467552284))]

In [8]:
reduced_embeddings = umap_model.fit_transform(embeddings)

fig = topic_model.visualize_documents(
    titles,
    reduced_embeddings=reduced_embeddings,
    width=1800,
    hide_annotations=True
)

fig.update_layout(font=dict(size=12))

In [9]:
topic_model.visualize_barchart(title=titles, n_words=10, autoscale=True, top_n_topics=4)

# topic_model.visualize_heatmap(n_clusters=30)

# topic_model.visualize_hierarchy()

In [10]:
from bertopic.representation import KeyBERTInspired

representation_model = KeyBERTInspired()
topic_model.update_topics(abstracts, representation_model=representation_model)

topic_model.visualize_barchart(title=titles, n_words=10, autoscale=True, top_n_topics=4)
# topic_model.get_topic_info()

In [11]:
from bertopic.representation import MaximalMarginalRelevance

representation_model = MaximalMarginalRelevance(diversity=0.2)
topic_model.update_topics(abstracts, representation_model=representation_model)

topic_model.visualize_barchart(title=titles, n_words=10, autoscale=True, top_n_topics=4)

In [28]:
import os
os.environ["http_proxy"]="127.0.0.1:1080"

import openai
from bertopic.representation import OpenAI

from bertopic.representation import _openai

_openai.DEFAULT_CHAT_PROMPT

prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short topic label in the following
format:
topic: <short topic label>
"""

client = openai.OpenAI(base_url="http://localhost:11434/v1", api_key="any")
representation_model = OpenAI(
    client, model="gemma4:e4b", exponential_backoff=True, chat=True, prompt=prompt,
    generator_kwargs={"stop": "xxxxx"}
)
topic_model.update_topics(abstracts, representation_model=representation_model)

  0%|          | 0/154 [00:03<?, ?it/s]


KeyboardInterrupt: 

In [32]:
topic_model.get_topic_info()

topic_model.visualize_barchart(title=titles, n_words=10, autoscale=True, top_n_topics=4)

KeyboardInterrupt: 